# 01 — Ingest SEC vào Bronze
Chạy với 4 ZIP thật hoặc fixture riêng. Không trộn hai nguồn. Bronze giữ trường gốc dưới dạng chuỗi có schema tường minh; chưa trim, cast hay khử trùng.

In [ ]:
from pathlib import Path
import os
import sys

candidate = Path(os.environ.get("SEC_PROJECT_ROOT", Path.cwd())).resolve()
locations = (Path.cwd().resolve(), Path.cwd().resolve().parent, candidate)
code_root = next((p for p in locations if (p / "src").is_dir()), None)
if code_root is None:
    raise RuntimeError("Hãy mở notebook từ sec_spark_project hoặc thư mục notebooks của dự án")
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
os.environ.setdefault("SEC_PROJECT_ROOT", str(code_root))
from src.config import make_spark, DATA, REPORTS, QUARTERS, TABLES
spark = make_spark()
print("Spark", spark.version, "|", spark.sparkContext.master, "|", QUARTERS)


## Kiểm tra cấu hình
Tệp SEC có thể dùng đuôi `.tsv` hoặc `.txt`, đều phân tách tab. Hàm ingest kiểm tra header, giải nén chọn 4 bảng và fail-fast khi cấu trúc sai. Tải thủ công ZIP vào `data/downloads` nếu chưa đặt `SEC_USER_AGENT`.

In [ ]:
from src.schemas import RAW_SCHEMAS, TARGET_TYPES
for table in TABLES:
    print(table, len(RAW_SCHEMAS[table].fields), TARGET_TYPES[table])

## Tải/giải nén/đọc/ghi
Mã đầy đủ các hàm tải và giải nén ở `src/ingest.py`. Mỗi quý được xử lý lần lượt và ghi Parquet sớm.

In [ ]:
from src.ingest import download_quarter, extract_quarter, read_txt, ingest_all
manifest = ingest_all(spark)
manifest

## Xác minh Bronze
Action `count()` đã thực hiện khi sinh manifest. Chỉ đọc vài dòng để xem, không collect bảng NUM.

In [ ]:
spark.read.parquet(str(DATA / "bronze" / "num" / QUARTERS[0])).printSchema()
spark.read.parquet(str(DATA / "bronze" / "num" / QUARTERS[0])).show(5, truncate=60)

In [ ]:
spark.stop()